# 01. Natural LLM
이 노트북은 `natural_llm` 모드를 단독으로 실험합니다.

프로젝트 경로를 설정하고 STREAM 공통 모듈을 import합니다.

In [5]:
from pathlib import Path
import sys

WORKDIR = Path.cwd()
REPO_ROOT = WORKDIR.parent if WORKDIR.name == "notebooks" else WORKDIR
if REPO_ROOT.name == "Root_Stream":
    REPO_ROOT = REPO_ROOT.parent
PROJECT_ROOT = REPO_ROOT / "Root_Stream"

for path_item in (REPO_ROOT, PROJECT_ROOT):
    if str(path_item) not in sys.path:
        sys.path.append(str(path_item))

from Root_Stream.utils.config_loader import load_config
from Root_Stream.utils.path_utils import resolve_path
from Root_Stream.prompts.prompt_manager import PromptManager
from Root_Stream.orchestrator.stream_orchestrator import StreamOrchestrator


사용자 질문을 입력합니다.

In [6]:
user_question = "SELECT TOP 50 * FROM INFORMATION_SCHEMA.TABLES"
user_question


'SELECT TOP 50 * FROM INFORMATION_SCHEMA.TABLES'

LLM에 질문을 전달해 SQL 문자열(query)을 생성합니다.

In [7]:
config_path = PROJECT_ROOT / "config" / "config.yaml"
config = load_config(config_path)
project_root = resolve_path(config.get("paths", {}).get("project_root", "."), PROJECT_ROOT)
prompt_file = resolve_path(config["paths"]["prompt_file"], project_root)
prompt_manager = PromptManager(prompt_file)

config["mode"] = "natural_llm"

system_prompt = prompt_manager.get_prompt("default_system_prompt")
final_prompt = user_question.strip()
print("prompt_key: default_system_prompt")
print("----- SYSTEM PROMPT -----")
print(system_prompt)
print("----- FINAL USER PROMPT -----")
print(final_prompt)

orchestrator = StreamOrchestrator(config=config, prompt_manager=prompt_manager, project_root=project_root)
result = orchestrator.run(user_question)
result.to_dict()


[2026-04-07 15:19:32] [INFO] [config_loader.py:load_config:38] 설정 로드 시작: c:\Users\김민한\Desktop\개발\DB_to_LLM\Root_Stream\config\config.yaml
[2026-04-07 15:19:32] [INFO] [config_loader.py:load_config:51] 설정 로드 완료
[2026-04-07 15:19:32] [INFO] [prompt_manager.py:_load_prompts:57] 프롬프트 파일 로드 시작: C:\Users\김민한\Desktop\개발\DB_to_LLM\Root_Stream\prompts\prompt_templates.yaml
[2026-04-07 15:19:32] [INFO] [prompt_manager.py:_load_prompts:70] 프롬프트 파일 로드 완료: prompt_count=4
[2026-04-07 15:19:32] [INFO] [stream_orchestrator.py:run_request:110] STREAM 실행 시작: mode=natural_llm
[2026-04-07 15:19:32] [INFO] [mode_natural_llm.py:run_natural_llm_mode:65] mode 실행 시작: natural_llm
[2026-04-07 15:19:32] [INFO] [ollama_client.py:generate:122] Ollama 호출 시작: model=qwen2.5:14b, endpoint=http://192.168.0.74:11434/api/chat


prompt_key: default_system_prompt
----- SYSTEM PROMPT -----
너는 Microsoft SQL Server(MSSQL) 전용 Text-to-SQL 질의 생성 전문가다.
사용자의 자연어 질문을 분석하여 실제 실행 가능한 MSSQL SELECT 쿼리를 생성하는 것이 주 임무다.

[핵심 역할]
- 사용자의 질문 의도를 정확히 해석한다.
- 제공된 스키마 정보와 업무 규칙을 근거로 SQL을 작성한다.
- 실행 가능성, 안전성, 일관성을 우선한다.
- 설명문보다 최종 SQL 자체의 정확성을 최우선으로 한다.

[절대 원칙]
1. 반드시 MSSQL 문법만 사용한다.
2. 반드시 SELECT 쿼리만 생성한다.
3. INSERT, UPDATE, DELETE, MERGE, DROP, ALTER, TRUNCATE, CREATE, EXEC 등 변경 구문은 절대 생성하지 않는다.
4. 제공되지 않은 테이블명, 컬럼명, 스키마명을 임의로 만들어내지 않는다.
5. 질문이 애매하더라도 과도한 추측을 하지 않는다.
6. 출력은 반드시 SQL 본문만 반환한다.
7. 설명, 코드블록, 주석, JSON, YAML은 포함하지 않는다.
8. 하나의 응답에는 하나의 SQL statement만 반환한다.

[MSSQL 규칙]
- LIMIT 대신 TOP을 사용한다.
- 날짜 계산은 DATEADD, DATEDIFF를 사용한다.
- 현재 시각 기준은 GETDATE()를 사용한다.
- 날짜 변환은 CAST 또는 CONVERT를 사용한다.
- TOP 사용 시 필요한 경우 ORDER BY를 함께 사용한다.
- 집계 쿼리에서는 비집계 컬럼이 모두 GROUP BY에 포함되도록 한다.

[출력 형식]
- SQL 본문만 반환한다.
- 설명하지 않는다.
- 코드블록을 사용하지 않는다.

----- FINAL USER PROMPT -----
SELECT TOP 50 * FROM INFORMATION_SCHEMA.TABLES


[2026-04-07 15:19:33] [INFO] [ollama_client.py:generate:170] Ollama 호출 완료: output_length=46
[2026-04-07 15:19:33] [INFO] [mode_natural_llm.py:run_natural_llm_mode:97] mode 실행 완료: natural_llm
[2026-04-07 15:19:33] [INFO] [stream_orchestrator.py:run_request:175] STREAM 실행 완료: mode=natural_llm


{'mode': 'natural_llm',
 'question': 'SELECT TOP 50 * FROM INFORMATION_SCHEMA.TABLES',
 'query': 'SELECT TOP 50 * FROM INFORMATION_SCHEMA.TABLES',
 'llm_provider': 'ollama',
 'prompt_key': 'default_system_prompt',
 'retrieved_contexts': [],
 'raw_response': None,
 'metadata': {'mode': 'natural_llm', 'llm_provider': 'ollama'}}

## MSSQL 실행 단계
위에서 생성된 `result.query`를 공통 SQL 실행 서비스로 안전하게 실행합니다.

In [8]:
from IPython.display import display

from Root_Stream.services.sql.sql_executor_service import SqlExecutorService
from Root_Stream.services.sql.sql_execution_integration import (
    build_execution_payload,
    run_generated_sql_with_executor,
)

# MSSQL connection (IP / PORT / DB / ID / PW)
db_host = "192.168.0.11"
db_port = 1433
db_name = "SVM3PRISM"
db_user = "prismadm"
db_password = "prism@2025"

database_config = config.setdefault("database", {})
database_config.update(
    {
        "type": "mssql",
        "host": db_host,
        "port": db_port,
        "database": db_name,
        "username": db_user,
        "password": db_password,
        "driver": database_config.get("driver", "ODBC Driver 17 for SQL Server"),
        "encrypt": bool(database_config.get("encrypt", False)),
        "trust_server_certificate": bool(database_config.get("trust_server_certificate", True)),
        "timeout": int(database_config.get("timeout", 30)),
    }
)

sql_config = config.setdefault("sql", {})
sql_config.setdefault("allow_only_select", True)
sql_config.setdefault("max_rows", 1000)

if "result" not in locals():
    raise ValueError("먼저 SQL 생성 셀을 실행해 result를 만든 뒤 이 셀을 실행하세요.")

executor = SqlExecutorService.from_config(config)
try:
    generated_sql = result.query
    df = run_generated_sql_with_executor(generated_sql=generated_sql, executor=executor)
    execution_payload = build_execution_payload(df)
finally:
    executor.close()

print(f"조회 row_count: {execution_payload['row_count']}")
print("조회 결과 표(최대 20행 미리보기)")
display(df.head(20))
execution_payload


[2026-04-07 15:19:36] [INFO] [sql_executor_service.py:from_config:59] SQL 실행 서비스 생성 완료: allow_only_select=True, max_rows=1000
[2026-04-07 15:19:36] [INFO] [sql_execution_integration.py:run_generated_sql_with_executor:44] 생성 SQL 실행 연결 시작
[2026-04-07 15:19:36] [INFO] [sql_executor_service.py:run_query:81] SQL 실행 요청 수신
[2026-04-07 15:19:36] [INFO] [sql_guard.py:validate_query_sql:59] SQL 검증 시작
[2026-04-07 15:19:36] [INFO] [sql_guard.py:validate_query_sql:73] SQL 검증 완료
[2026-04-07 15:19:36] [INFO] [sql_executor_service.py:run_query:83] MSSQL 조회 실행 시작
[2026-04-07 15:19:36] [INFO] [mssql_client.py:execute_query:110] MSSQL 조회 시작: host=192.168.0.11, database=SVM3PRISM
[2026-04-07 15:19:36] [INFO] [mssql_client.py:execute_query:119] MSSQL 조회 완료: row_count=50
[2026-04-07 15:19:36] [INFO] [sql_executor_service.py:run_query:86] SQL 실행 완료: row_count=50
[2026-04-07 15:19:36] [INFO] [sql_execution_integration.py:run_generated_sql_with_executor:46] 생성 SQL 실행 연결 완료: row_count=50
[2026-04-07 15:19:36] [

?? row_count: 50
?? ?? ?(?? 20? ????)


,TABLE_CATALOG,TABLE_SCHEMA,TABLE_NAME,TABLE_TYPE
0,SVM3PRISM,dbo,LOG_SAVE_MST_CUBIC,BASE TABLE
1,SVM3PRISM,dbo,sysdiagrams,BASE TABLE
2,SVM3PRISM,dbo,LOG_SAVE_MST_PLM,BASE TABLE
3,SVM3PRISM,dbo,LogErrorDef,BASE TABLE
4,SVM3PRISM,dbo,MACHINESPEC,BASE TABLE
5,SVM3PRISM,dbo,MCC_RAW_DATA2,BASE TABLE
6,SVM3PRISM,dbo,MccBaseStep,BASE TABLE
7,SVM3PRISM,dbo,MccEqpInfo,BASE TABLE
8,SVM3PRISM,dbo,A2PhtoNiconReloadConfig,BASE TABLE
9,SVM3PRISM,dbo,ACTION_LOG_DATA,BASE TABLE


{'columns': ['TABLE_CATALOG', 'TABLE_SCHEMA', 'TABLE_NAME', 'TABLE_TYPE'],
 'row_count': 50,
 'rows': [{'TABLE_CATALOG': 'SVM3PRISM',
   'TABLE_SCHEMA': 'dbo',
   'TABLE_NAME': 'LOG_SAVE_MST_CUBIC',
   'TABLE_TYPE': 'BASE TABLE'},
  {'TABLE_CATALOG': 'SVM3PRISM',
   'TABLE_SCHEMA': 'dbo',
   'TABLE_NAME': 'sysdiagrams',
   'TABLE_TYPE': 'BASE TABLE'},
  {'TABLE_CATALOG': 'SVM3PRISM',
   'TABLE_SCHEMA': 'dbo',
   'TABLE_NAME': 'LOG_SAVE_MST_PLM',
   'TABLE_TYPE': 'BASE TABLE'},
  {'TABLE_CATALOG': 'SVM3PRISM',
   'TABLE_SCHEMA': 'dbo',
   'TABLE_NAME': 'LogErrorDef',
   'TABLE_TYPE': 'BASE TABLE'},
  {'TABLE_CATALOG': 'SVM3PRISM',
   'TABLE_SCHEMA': 'dbo',
   'TABLE_NAME': 'MACHINESPEC',
   'TABLE_TYPE': 'BASE TABLE'},
  {'TABLE_CATALOG': 'SVM3PRISM',
   'TABLE_SCHEMA': 'dbo',
   'TABLE_NAME': 'MCC_RAW_DATA2',
   'TABLE_TYPE': 'BASE TABLE'},
  {'TABLE_CATALOG': 'SVM3PRISM',
   'TABLE_SCHEMA': 'dbo',
   'TABLE_NAME': 'MccBaseStep',
   'TABLE_TYPE': 'BASE TABLE'},
  {'TABLE_CATALOG': 'SVM